## Example 1: SSH client setup and key management

In [ ]:
import paramiko
import io

# Example 1: SSH Client Basic Setup and Key Management
try:
    # Create SSH client instance
    client = paramiko.SSHClient()
    print(f"SSH client created: {type(client).__name__}")
    
    # Set missing host key policy
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    print("Host key policy set to AutoAddPolicy")
    
    # Generate RSA key for demonstration
    key = paramiko.RSAKey.generate(1024)
    print(f"Generated RSA key: {key.get_name()}")
    print(f"Key size: {key.size}")
    print(f"Key fingerprint: {key.get_fingerprint().hex()[:16]}...")
    
    # Key serialization
    key_str = io.StringIO()
    key.write_private_key(key_str)
    print(f"Private key serialized (length: {len(key_str.getvalue())} chars)")
    assert key.size == 1024
    
    # Reload the serialized key and check it is the same key
    reloaded = paramiko.RSAKey.from_private_key(io.StringIO(key_str.getvalue()))
    assert reloaded.get_base64() == key.get_base64()
    print("Private key reloaded from PEM text: same public key")
    
    # Show available authentication methods
    auth_methods = ['password', 'publickey', 'keyboard-interactive']
    print(f"Common auth methods: {auth_methods}")
    
    print("Example 1: SSH client setup and key management completed")
    
except Exception as e:
    print(f"Error in SSH client setup: {e}")
    raise AssertionError(
        f"paramiko example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Key types and SSH config

In [ ]:
import paramiko
import io

# Example 2: Key types and SSH config
print("Example 2 - Key types and config")

# Generate RSA key
rsa_key = paramiko.RSAKey.generate(1024)
print(f"RSA key: {rsa_key.get_name()}, size: {rsa_key.size}")

# Show available key types
print(f"Available key classes: RSAKey, ECDSAKey, Ed25519Key")

# SSH config parsing
config = paramiko.SSHConfig()
config_data = '''
Host example
    HostName example.com
    User testuser
    Port 2222
'''
config.parse(io.StringIO(config_data))
host_config = config.lookup('example')
print(f"Parsed config: hostname={host_config['hostname']}, user={host_config['user']}")
assert host_config['hostname'] == 'example.com'
assert host_config['user'] == 'testuser'
assert host_config['port'] == '2222'

# Show transport preferences
print(f"Preferred KEX: {paramiko.Transport._preferred_kex[:2]}")
print(f"Preferred ciphers: {paramiko.Transport._preferred_ciphers[:3]}")

print("Example 2 completed")

## Example 3: Key operations

In [ ]:
import paramiko

# Example 3: Key operations
print("Example 3 - Key operations")

# Generate RSA key
private_key = paramiko.RSAKey.generate(2048)
print(f"Generated 2048-bit RSA key")

# Public key format
public_key = f"{private_key.get_name()} {private_key.get_base64()}"
print(f"Public key: {public_key[:50]}...")

# Fingerprint (MD5)
fingerprint = private_key.get_fingerprint()
print(f"MD5 fingerprint: {fingerprint.hex()}")

# Sign data and verify the signature with the same key
assert private_key.size == 2048
signature = private_key.sign_ssh_data(b"smoke test payload")
signature.rewind()
assert private_key.verify_ssh_sig(b"smoke test payload", signature)
signature.rewind()
assert not private_key.verify_ssh_sig(b"tampered payload", signature)
print("Signature verified; a tampered payload is rejected")

# Compare keys
another_key = paramiko.RSAKey.generate(2048)
print(f"Keys different: {private_key.get_base64() != another_key.get_base64()}")

# Host keys
host_keys = paramiko.HostKeys()
host_keys.add("example.com", private_key.get_name(), private_key)
print(f"Added host key for example.com")
print(f"Host keys count: {len(host_keys)}")
assert len(host_keys) == 1
assert host_keys.lookup("example.com")[private_key.get_name()] == private_key

print("Example 3 completed")

## Example 4: Transport layer and protocol configuration

In [ ]:
import paramiko
import socket

# Example 4: Transport Layer and Protocol Configuration
try:
    # Demonstrate transport configuration without actual connection
    print("Transport layer configuration:")
    
    # Show security preferences
    print("Security algorithms:")
    print(f"Preferred KEX: {paramiko.Transport._preferred_kex[:3]}")
    print(f"Preferred server keys: {paramiko.Transport._preferred_keys[:3]}")
    print(f"Preferred ciphers: {paramiko.Transport._preferred_ciphers[:3]}")
    print(f"Preferred MACs: {paramiko.Transport._preferred_macs[:3]}")
    
    # Create mock socket for transport demo
    # (In real use, this would be connected to SSH server)
    class MockSocket:
        def __init__(self):
            self.closed = False
        def close(self):
            self.closed = True
        def settimeout(self, timeout):
            pass
    
    mock_sock = MockSocket()
    
    # Show transport creation (without actual connection)
    print(f"Mock socket created: {type(mock_sock).__name__}")
    print(f"Socket closed: {mock_sock.closed}")
    
    # Authentication methods enumeration
    auth_types = {
        'none': 'No authentication',
        'password': 'Password authentication', 
        'publickey': 'Public key authentication',
        'keyboard-interactive': 'Interactive authentication',
        'gssapi-with-mic': 'GSSAPI authentication'
    }
    
    print(f"\nAuthentication methods:")
    for method, desc in auth_types.items():
        print(f"- {method}: {desc}")
    
    # Exception types that paramiko can raise
    exceptions = [paramiko.SSHException, paramiko.AuthenticationException, 
                 paramiko.BadHostKeyException, paramiko.ChannelException]
    print(f"\nParamiko exception types: {[e.__name__ for e in exceptions]}")
    assert all(issubclass(exc, paramiko.SSHException) for exc in exceptions)
    
    print("Example 4: Transport and protocol configuration completed")
    
except Exception as e:
    print(f"Error in transport configuration: {e}")
    raise AssertionError(
        f"paramiko example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Configuration files and agent integration

In [ ]:
import io

import paramiko

# Example 5: Configuration Files and Agent Integration
try:
    # SSH configuration file parsing and management
    print("SSH configuration management:")
    
    # Create temporary SSH config for demonstration
    config_content = '''
# SSH Client Configuration
Host production
    HostName prod.example.com
    User deploy
    Port 22
    IdentityFile ~/.ssh/id_rsa
    ForwardAgent yes

Host staging
    HostName staging.example.com
    User testuser
    Port 2222
    IdentityFile ~/.ssh/id_ed25519
    
Host *.example.com
    User defaultuser
    Compression yes
    ServerAliveInterval 60
    '''
    
    # Parse configuration
    ssh_config = paramiko.SSHConfig()
    ssh_config.parse(io.StringIO(config_content))
    
    # Test configuration lookup
    hosts = ['production', 'staging', 'other.example.com']
    for host in hosts:
        config = ssh_config.lookup(host)
        print(f"\nConfig for '{host}':")
        print(f"  Hostname: {config.get('hostname', 'not set')}")
        print(f"  User: {config.get('user', 'not set')}")
        print(f"  Port: {config.get('port', 'not set')}")
        print(f"  IdentityFile: {config.get('identityfile', ['not set'])[0] if config.get('identityfile') else 'not set'}")
    
    production = ssh_config.lookup('production')
    staging = ssh_config.lookup('staging')
    other = ssh_config.lookup('other.example.com')
    assert production['hostname'] == 'prod.example.com' and production['user'] == 'deploy'
    assert staging['port'] == '2222'
    assert other['user'] == 'defaultuser' and other['compression'] == 'yes'
    
    # SSH agent client; without a running agent it simply reports no keys
    agent = paramiko.Agent()
    print(f"\nSSH agent interface created: {type(agent).__name__}")
    agent_keys = agent.get_keys()
    assert isinstance(agent_keys, tuple)
    print(f"Keys offered by the SSH agent: {len(agent_keys)}")
    agent.close()
    
    # Show paramiko version and capabilities
    print(f"\nParamiko version: {paramiko.__version__}")
    print(f"Available classes: {[cls for cls in dir(paramiko) if cls[0].isupper()][:5]}...")
    
    print("Example 5: Configuration and agent integration completed")
    
except Exception as e:
    print(f"Error in configuration management: {e}")
    raise AssertionError(
        f"paramiko example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e